In [1]:
import os
import json
import gzip

from tqdm import tqdm

FILE_PATHS = [
    '/home/v-murongma/code/OpenHands_SWE-Bench-Optimized/evaluation/evaluation_outputs/outputs/SWE-Gym__SWE-Gym-train/CodeActAgent/Qwen3-Coder-480B-A35B-Instruct_maxiter_100_N_v0.61.0-no-hint-train-qwen3_coder_480b_a35b_instruct-t05/Qwen3-Coder-480B-A35B-Instruct_maxiter_100_N_v0.61.0-no-hint-train-qwen3_coder_480b_a35b_instruct-t05-run_1/output.with_completions.jsonl.gz',
    # '/home/v-murongma/code/OpenHands_SWE-Bench-Optimized/evaluation/evaluation_outputs/outputs/SWE-Gym__SWE-Gym-train/CodeActAgent/Qwen3-Coder-480B-A35B-Instruct_maxiter_100_N_v0.61.0-no-hint-train-qwen3_coder_480b_a35b_instruct-t05/Qwen3-Coder-480B-A35B-Instruct_maxiter_100_N_v0.61.0-no-hint-train-qwen3_coder_480b_a35b_instruct-t05-run_2/output.with_completions.jsonl.gz',
]

data=[]
for file_path in FILE_PATHS:
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc=f"Processing {os.path.basename(file_path)}"):
            data.append(json.loads(line))
print(f"Total records loaded: {len(data)}")



Processing output.with_completions.jsonl.gz: 2119it [00:20, 102.43it/s]

Total records loaded: 2119


In [2]:
data[0].keys()

dict_keys(['instance_id', 'test_result', 'instruction', 'metadata', 'history', 'metrics', 'error', 'instance', 'report', 'raw_completions'])

In [6]:
data[0]['test_result']["git_patch"]

'diff --git a/moto/timestreamwrite/models.py b/moto/timestreamwrite/models.py\nindex bb87c3cba..3cf25e151 100644\n--- a/moto/timestreamwrite/models.py\n+++ b/moto/timestreamwrite/models.py\n@@ -131,6 +131,17 @@ class TimestreamWriteBackend(BaseBackend):\n         database = self.describe_database(database_name)\n         table = database.describe_table(table_name)\n         table.write_records(records)\n+        \n+        # Return ingestion statistics\n+        # For simplicity in mock, assume all records go to MemoryStore\n+        total_records = len(records)\n+        return {\n+            "RecordsIngested": {\n+                "Total": total_records,\n+                "MemoryStore": total_records,\n+                "MagneticStore": 0\n+            }\n+        }\n \n     def describe_endpoints(self):\n         # https://docs.aws.amazon.com/timestream/latest/developerguide/Using-API.endpoint-discovery.how-it-works.html\ndiff --git a/moto/timestreamwrite/responses.py b/moto/timestre

In [8]:
data[0]['instance'].keys()

dict_keys(['instance_id', 'hints_text', 'patch', 'test_patch', 'created_at', 'problem_statement', 'repo', 'base_commit', 'version', 'PASS_TO_PASS', 'FAIL_TO_PASS'])

In [19]:
def inspect_with_id(data, id):
    d = data[id]
    print(f"Inspecting instance {d['instance_id']}")
    print("Problem Statement:")
    print(d['instance']['problem_statement'])
    print("Golden Code Fix Patch:")
    print(d['instance']['patch'])
    print("Golden Test Patch:")
    print(d['instance']['test_patch'])
    print("Generated Patch Result:")
    print(d['test_result'])
    print("Test Report:")
    print(d['report'])

inspect_with_id(data, 7)

Inspecting instance getmoto__moto-7456
Problem Statement:
Please add support for the ResilienceHub API
I am using moto 5.0.2, Python mocks with Python 3.11.3, boto3 1.26.68, botocore 1.29.165.

I have the following code:
```
@pytest.fixture(scope="function")
def aws_credentials():
    """Mocked AWS Credentials for moto."""
    os.environ["AWS_ACCESS_KEY_ID"] = "testing"
    os.environ["AWS_SECRET_ACCESS_KEY"] = "testing"
    os.environ["AWS_SECURITY_TOKEN"] = "testing"
    os.environ["AWS_SESSION_TOKEN"] = "testing"
    os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

@pytest.fixture(scope="function")
def resiliencehub(aws_credentials):
    with mock_aws():
        yield boto3.client("resiliencehub", region_name="us-east-2")

@pytest.fixture
def create_policy(resiliencehub):
    response = boto3.client("resiliencehub").create_resiliency_policy(
        policyName="mock-resilience-hub-basic-webapp",
        policyDescription="Mocked resiliency policy",
        policy={"AZ": {"rpoInSecs":